[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r2/r2-q1/03_baseline_classifier.ipynb)

# R2-Q1 Week 2 — Train a baseline plant disease classifier

This notebook trains a convolutional neural network to classify plant
disease photographs. Your training data is **PlantVillage** — the
lab-condition image dataset you met in Notebook 01. You'll evaluate
the trained model on PlantVillage's own held-out test split.

By the end of this notebook you will have:

- A trained ResNet-18 classifier saved to `baseline_resnet18.pt`.
- A single accuracy number for how well the model does on PlantVillage's
  test split — your EQ#2 deliverable, saved to `eq2_results.json`.
- Training and validation loss curves saved to `training_curves.png`,
  so you can see how the model learned.

The point of this week is to confirm your classifier learned the lab
images well. If it didn't — if PV-internal accuracy is low — there's
no transfer test to run next week. A weak model that does badly on
PlantDoc tells you nothing, because it might be doing badly for
reasons that have nothing to do with the lab-to-field shift.

This notebook uses a GPU. Before you run anything, switch your Colab
runtime to a T4 GPU (Runtime → Change runtime type → T4 GPU → Save).
Setup Step 2 will check for the GPU and stop with a clear error if
one isn't attached.

In [ ]:
try:
    import irilab2026 as iri
    print(f"OK — irilab2026 {iri.__version__} is installed and all deps importable.")
    print("    → use Pattern 2 in Cell 1:")
    print("      !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main")
except ModuleNotFoundError as e:
    print(f"MISSING — {e.name} not importable.")
    print("    → use Pattern 1 in Cell 1:")
    print("      !pip install git+https://github.com/geraldmc/irilab2026.git@main")

In [ ]:
# Pattern 1 — fresh or partial runtime (installs deps that aren't present yet)
# This skips anything pip already sees as installed. This is ideal for a new environment or when you want 
# to be sure everything is up to date.

# !pip install git+https://github.com/geraldmc/irilab2026.git@main

# Pattern 2 — populated runtime (refreshes irilab2026 only, leaves deps alone)
# This is ideal for when you already have the dependencies installed and just want to update irilab2026.

# !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main

**Before you run the next cell: switch your runtime to GPU.**

This notebook trains a neural network, which needs a GPU. By default,
Colab gives you a CPU-only runtime — fine for last week's notebooks,
but it won't work for this one. The next cell will refuse to run
until you switch.

To switch the runtime:

1. In Colab's menu bar: **Runtime → Change runtime type**.
2. Under "Hardware accelerator," choose **T4 GPU**.
3. Click **Save**.

Colab will restart the session, which means anything you've already
run in this notebook will be cleared. That's fine — you've only run
the install cell so far, and pip has cached the install so a re-run
takes seconds, not minutes. After the restart, re-run Setup Step 1
(the install cell above) and then this cell.

**If Colab tells you "no GPUs available right now":** free-tier GPU
access is shared and occasionally unavailable. Wait a few minutes
and try the menu change again. If it keeps refusing, you have two
options: wait longer, or tell your mentor and they'll help you find
an alternative (the upgraded Colab tiers have more reliable GPU
access).

**Once you've successfully switched:** the cell below will print a
one-line summary ending in `GPU OK` and you can continue to
Section 2.

In [ ]:
# Mount Drive, set up irilab2026 with a GPU requirement, seed everything,
# and point OUTPUT_DIR at the R2-Q1 output directory.
import irilab2026 as iri
iri.setup(gpu_required=True)
iri.seed_all(42)

OUTPUT_DIR = iri.output_dir("r2_q1")
print(f"Output directory: {OUTPUT_DIR}")

# DEV_MODE switches the notebook between fast-iteration mode (tiny PV
# variant, 2 epochs — runs end to end in ~3 minutes on a T4) and the
# real production run (full PV variant, 10 epochs — ~25 minutes).
#
# Use DEV_MODE = True while you're debugging or making changes. Switch
# to False for the run that produces your real EQ#2 number.
DEV_MODE = False

if DEV_MODE:
    PV_VARIANT = "tiny"
    EPOCHS = 2
    print("DEV_MODE ON: PV variant = 'tiny', epochs = 2")
else:
    PV_VARIANT = "full"
    EPOCHS = 10
    print("DEV_MODE OFF: PV variant = 'full', epochs = 10")

## 2) Verify your Week 1 file is here

At the end of Notebook 02 you saved a file capturing the three
decisions you committed to before Week 2: your aggregation level,
your class-space remapping, and your statistical test. That file is
your Week 1 deliverable.

You do not need its contents for this week's work. Week 2 is about
training a classifier and measuring its accuracy on PlantVillage's
own test split — none of that depends on the choices in your Week 1
file. Week 3 is when those choices start to matter.

The check below just confirms the file exists where Notebook 02 said
it would. If you reach Week 3 without it, you'll have a problem that
should be caught now rather than then.

In [ ]:
### 2.1) Confirm the file from Notebook 02 is in OUTPUT_DIR
from pathlib import Path

week1_file = OUTPUT_DIR / "precommit.json"
assert week1_file.exists(), (
    f"Couldn't find the file you saved at the end of Notebook 02. "
    f"Expected at: {week1_file}. "
    f"Re-run Notebook 02's final section to produce it."
)
print(f"Found your Week 1 file: {week1_file}")
print(f"  size: {week1_file.stat().st_size:,} bytes")

## 3) Load PlantVillage and build the splits

You'll work with three splits of the data this week:

- **train** — what the model learns from.
- **val** — a small slice held out from train, used during training
  to track how the model is doing on data it hasn't trained on. You
  don't touch test during training, so val is what tells you whether
  to keep training or stop.
- **test** — held in reserve until Section 7. This is what your
  reported PV-internal accuracy is computed on.

PlantVillage already has a `split` column in its metadata that marks
each image as `train` or `test`. You'll use that directly. To get a
val split, you'll carve a 10% stratified slice out of train —
"stratified" meaning each class contributes 10% of its train
images to val, so val has the same class proportions as train.

The seeded random state from Setup Step 2 makes the val carve
reproducible: re-running this section gives the same val rows every
time.

In [ ]:
### 3.1) Load metadata and the HF Dataset
metadata, hf_dataset = iri.load_plantvillage(variant=PV_VARIANT)

print(f"Loaded variant '{PV_VARIANT}'")
print(f"  rows:    {len(metadata):,}")
print(f"  classes: {metadata['class_label'].nunique()}")
print(f"  splits:  {dict(metadata['split'].value_counts())}")

PlantVillage's `split` column gives you train and test directly. The
cell below splits train into train-without-val and val using
scikit-learn's `train_test_split` with `stratify=class_label`, so
val ends up with the same class proportions as train.

The val fraction (`VAL_FRACTION = 0.1`) is a notebook-level constant
so a mentor can adjust it without hunting for the value inside the
code.

In [ ]:
### 3.2) Carve a stratified val split from train
from sklearn.model_selection import train_test_split

VAL_FRACTION = 0.1

train_all = metadata[metadata["split"] == "train"]
test_df = metadata[metadata["split"] == "test"]

train_df, val_df = train_test_split(
    train_all,
    test_size=VAL_FRACTION,
    stratify=train_all["class_label"],
    random_state=42,
)

print(f"train: {len(train_df):>6,} rows")
print(f"val:   {len(val_df):>6,} rows")
print(f"test:  {len(test_df):>6,} rows")
print()
print(f"total: {len(train_df) + len(val_df) + len(test_df):>6,} rows")
print(f"       (should match metadata length: {len(metadata):,})")

The model needs to know how many output classes to predict. Read this
from the data rather than hardcoding 38 — if you're in DEV_MODE on
the tiny variant, the class count is still 38, but reading it from
the data makes the code robust to future variants that might subset
classes.

In [ ]:
### 3.3) Number of classes
NUM_CLASSES = metadata["class_idx"].nunique()
print(f"Number of classes: {NUM_CLASSES}")

## 4) Inspect the splits before you train

Before you start training — which takes real time on a GPU — confirm
the splits look right. Two checks:

- **Class balance.** Each split should have all 38 classes. If val or
  test is missing a class entirely (a stratification edge case), the
  model will silently never be evaluated on that class. Catch it now.
- **Sample images.** Visually confirm that the images look like
  PlantVillage images and not, say, blank tensors or corrupted JPEGs.
  PIL and the HF Dataset both have failure modes that show up only
  when you actually look.

In [ ]:
### 4.1) Class counts per split
import pandas as pd

splits = {"train": train_df, "val": val_df, "test": test_df}

counts = pd.DataFrame({
    name: df["class_label"].value_counts() for name, df in splits.items()
}).fillna(0).astype(int)

print(f"Classes present in each split:")
for name, df in splits.items():
    n_classes = df["class_label"].nunique()
    print(f"  {name}: {n_classes} / {NUM_CLASSES}")

if (counts == 0).any().any():
    missing = counts[(counts == 0).any(axis=1)]
    print(f"\nWARNING: some classes are missing from some splits:")
    print(missing)
else:
    print(f"\nAll {NUM_CLASSES} classes present in all three splits.")

Pull one image from each split. Use the seeded random state so the
specific images shown are the same every time you re-run this
notebook.

In [ ]:
### 4.2) One sample image per split
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, (name, df) in zip(axes, splits.items()):
    row = df.sample(1, random_state=42).iloc[0]
    img = hf_dataset[int(row.name)]["image"].convert("RGB")
    ax.imshow(img)
    ax.set_title(f"{name}\n{row['class_label']}", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 5) Build the model and the transform pipeline

Two pieces of machinery need to be set up before training can start:

- **The model.** A ResNet-18 pretrained on ImageNet, with its final
  classification layer replaced to output 38 classes instead of
  ImageNet's 1,000. `iri.build_baseline_model()` does this in one
  call. ResNet-18 is the field's default choice for fast,
  defensible plant-disease classification on Colab GPUs.

- **The transforms.** Each image gets resized, cropped, normalized,
  and (for train only) randomly flipped before going to the model.
  The training transform adds randomness so the model sees small
  variations of each image across epochs; the evaluation transform
  is deterministic so val and test results are reproducible. Both
  use the standard ImageNet normalization the pretrained weights
  expect.

You won't write the transform pipeline by hand — `irilab2026`
provides it. This isn't hiding important information: the transforms
are completely standard, and writing them out would be 10 lines of
boilerplate per notebook. The library exposes them once so notebooks
stay focused on the experiment.

In [ ]:
### 5.1) Build the model and move it to the GPU
import torch

device = torch.device("cuda")
model = iri.build_baseline_model(num_classes=NUM_CLASSES).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model:        ResNet-18, pretrained ImageNet, 38-class head")
print(f"Parameters:   {n_params:,}")
print(f"Device:       {device}")

In [ ]:
### 5.2) Build the transforms
train_transform = iri.imagenet_train_transform()
eval_transform = iri.imagenet_eval_transform()

# Print them so the student can see what's actually being applied.
print("Training transform:")
print(train_transform)
print()
print("Evaluation transform:")
print(eval_transform)

A **DataLoader** is PyTorch's batching mechanism. It takes a Dataset
(your `PlantVillageDataset` wrapper) and yields tensors of shape
`(batch_size, 3, 224, 224)` ready for the model. Train uses
`shuffle=True` so the model sees images in a different order each
epoch; val and test use `shuffle=False` because order doesn't matter
when you're not training.

`num_workers=2` runs two background processes to load images while
the GPU is busy with the previous batch. On Colab this saves real
time without using much extra memory.

In [ ]:
### 5.3) Build the DataLoaders
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_set = iri.PlantVillageDataset(train_df, hf_dataset, transform=train_transform)
val_set   = iri.PlantVillageDataset(val_df,   hf_dataset, transform=eval_transform)
test_set  = iri.PlantVillageDataset(test_df,  hf_dataset, transform=eval_transform)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"train_loader: {len(train_loader)} batches of {BATCH_SIZE}")
print(f"val_loader:   {len(val_loader)} batches of {BATCH_SIZE}")
print(f"test_loader:  {len(test_loader)} batches of {BATCH_SIZE}")

## 6) Train the model

Training a neural network is a loop. For each epoch (one full pass
through the training data):

1. Put the model in **train mode** so dropout and batch-norm layers
   behave correctly for learning.
2. For each batch of images, run a **forward pass** (compute the
   model's predictions), compute the **loss** (how wrong the
   predictions are), run a **backward pass** (compute how to adjust
   each parameter), and have the **optimizer** apply the adjustment.
3. After the epoch, put the model in **eval mode** and run it on
   val to see how it's doing on data it hasn't trained on.
4. If val accuracy went up, save this version of the model as the
   "best so far." If a later epoch beats it, that one replaces it.

The optimizer is SGD with momentum 0.9. The learning rate starts at
0.01 and drops to 0.001 after epoch 7 — a standard schedule for
fine-tuning a pretrained network. Loss function is cross-entropy,
the standard choice for multi-class classification.

You'll see a one-line summary printed at the end of each epoch:
train loss, val loss, val accuracy. Watch val accuracy go up; that's
the signal that learning is happening.

In [ ]:
### 6.1) Set up the optimizer, scheduler, and loss
import torch.nn as nn

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01,
    momentum=0.9,
)
scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer,
    milestones=[7],
    gamma=0.1,
)
criterion = nn.CrossEntropyLoss()

print(f"Optimizer:    SGD(lr=0.01, momentum=0.9)")
print(f"Scheduler:    step lr=0.01 → 0.001 at epoch 7")
print(f"Loss:         CrossEntropyLoss")
print(f"Epochs:       {EPOCHS}")

The next cell is the actual training loop. It will take a while: on a
Colab T4, expect ~3 minutes per epoch on the full PV variant. With
`EPOCHS = 10`, that's roughly 25-30 minutes of wall time.

Track which version of the model performs best on val and keep it.
At the end of the loop, the `model` variable in memory carries the
best-val weights — not the final epoch's weights, which may have
started to overfit.

In [ ]:
### 6.2) Train
import copy
import time

train_losses, val_losses, val_accs = [], [], []
best_val_acc = 0.0
best_state = copy.deepcopy(model.state_dict())

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()

    # --- train ---
    model.train()
    running_loss, n_seen = 0.0, 0
    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        n_seen += images.size(0)

    train_loss = running_loss / n_seen
    train_losses.append(train_loss)

    # --- validate ---
    model.eval()
    running_loss, n_correct, n_seen = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            logits = model(images)
            loss = criterion(logits, labels)
            running_loss += loss.item() * images.size(0)
            n_correct += (logits.argmax(dim=1) == labels).sum().item()
            n_seen += images.size(0)

    val_loss = running_loss / n_seen
    val_acc = n_correct / n_seen
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())
        marker = "  ← new best"
    else:
        marker = ""

    scheduler.step()

    elapsed = time.time() - epoch_start
    print(
        f"epoch {epoch:>2} | "
        f"train_loss {train_loss:.4f} | "
        f"val_loss {val_loss:.4f} | "
        f"val_acc {val_acc:.4f} | "
        f"{elapsed:5.1f}s{marker}"
    )

# Restore best-val weights for evaluation.
model.load_state_dict(best_state)
print(f"\nBest val accuracy: {best_val_acc:.4f}")
print(f"Restored best-val weights for test evaluation.")

In [ ]:
### 6.3) Plot the training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

epochs_x = list(range(1, EPOCHS + 1))

ax1.plot(epochs_x, train_losses, label="train")
ax1.plot(epochs_x, val_losses,   label="val")
ax1.set_xlabel("epoch")
ax1.set_ylabel("loss")
ax1.set_title("Loss")
ax1.legend()

ax2.plot(epochs_x, val_accs, label="val", color="tab:green")
ax2.set_xlabel("epoch")
ax2.set_ylabel("accuracy")
ax2.set_title("Validation accuracy")
ax2.set_ylim(0, 1)
ax2.legend()

plt.tight_layout()

curves_path = OUTPUT_DIR / "training_curves.png"
plt.savefig(curves_path, dpi=120, bbox_inches="tight")
plt.show()

print(f"Saved: {curves_path}")

## 7) Evaluate on the PV-internal test split

The trained model now runs on the test split — the rows PlantVillage
marked as `split == "test"` that you have not touched until this
point. The accuracy you compute here is your EQ#2 number.

Two things come out of this section:

- **Overall test accuracy.** A single number. This is what EQ#2 asks
  for, and what your go/no-go check uses.
- **A confusion matrix.** A 38 × 38 table showing which classes the
  model confused with which. You won't analyze this in detail this
  week — but if your accuracy is unexpectedly low, the confusion
  matrix is the first place to look for an explanation.

The **go/no-go threshold** for proceeding to Week 3 is overall
accuracy ≥ 95%. The R2-Q1 page's Prediction section says PV-internal
accuracy "likely exceeds 95%" on this kind of data. If your model
clears 95%, you have a reasonable lab-condition baseline and the
Week 3 transfer test is worth running. If it doesn't, training had
a problem and the transfer measurement won't tell you anything
useful — debug or talk to your mentor before moving on.

In [ ]:
### 7.1) Compute test predictions
import numpy as np

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)
        logits = model(images)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.append(preds)
        all_labels.append(labels.numpy())

all_preds = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)
print(f"Predictions computed for {len(all_preds):,} test images.")

In [ ]:
### 7.2) Overall test accuracy and go/no-go
GO_THRESHOLD = 0.95

test_acc = (all_preds == all_labels).mean()

print(f"PV-internal test accuracy: {test_acc:.4f} ({test_acc:.1%})")
print(f"Threshold for Week 3:      {GO_THRESHOLD:.1%}")
print()
if test_acc >= GO_THRESHOLD:
    verdict = "go"
    print(f"VERDICT: GO. Your classifier learned the lab data well enough")
    print(f"         to make the Week 3 transfer test meaningful.")
else:
    verdict = "no-go"
    print(f"VERDICT: NO-GO. Your classifier is below the threshold for a")
    print(f"         meaningful transfer test. Debug Section 6 (training)")
    print(f"         or talk to your mentor before continuing to Week 3.")

In [ ]:
### 7.3) Confusion matrix
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(all_labels, all_preds, labels=list(range(NUM_CLASSES)))

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(cm, cmap="Blues")
ax.set_xlabel("predicted class")
ax.set_ylabel("true class")
ax.set_title(f"Confusion matrix on PV test split (n = {len(all_labels):,})")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

# Off-diagonal totals: how many test images got the wrong class.
n_wrong = (all_preds != all_labels).sum()
print(f"Total misclassifications: {n_wrong:,} / {len(all_labels):,} ({n_wrong / len(all_labels):.2%})")

## 8) Save your trained model and your EQ#2 results

Two files get written here:

- `baseline_resnet18.pt` — the model's learned parameters
  (its "state dict"). Notebook 04 will load this directly so you
  don't have to retrain when you move to Week 3.
- `eq2_results.json` — your EQ#2 deliverable: overall test accuracy,
  the number of test images it was computed on, the threshold, and
  the go/no-go verdict.

The training-curves PNG was already saved in Section 6.

In [ ]:
### 8.1) Save the model state dict
model_path = OUTPUT_DIR / "baseline_resnet18.pt"
torch.save(model.state_dict(), model_path)

print(f"Saved model: {model_path}")
print(f"  size: {model_path.stat().st_size / 1e6:.1f} MB")

In [ ]:
### 8.2) Save the EQ#2 results
import json

eq2_results = {
    "test_accuracy": float(test_acc),
    "n_test_images": int(len(all_labels)),
    "go_threshold": float(GO_THRESHOLD),
    "verdict": verdict,
    "epochs_trained": int(EPOCHS),
    "best_val_accuracy": float(best_val_acc),
    "pv_variant": PV_VARIANT,
    "dev_mode": bool(DEV_MODE),
}

results_path = OUTPUT_DIR / "eq2_results.json"
with open(results_path, "w") as f:
    json.dump(eq2_results, f, indent=2)

print(f"Saved results: {results_path}")
print()
print("Contents:")
print(json.dumps(eq2_results, indent=2))

In [ ]:
### 8.3) Sanity-load
# Re-read both files to confirm they wrote cleanly.
state = torch.load(model_path, map_location="cpu", weights_only=True)
assert isinstance(state, dict), "Saved model is not a state dict"
print(f"Reloaded model state dict: {len(state)} tensors")

with open(results_path) as f:
    reloaded = json.load(f)
assert reloaded == eq2_results, "Reloaded EQ#2 results differ from in-memory"
print(f"Reloaded EQ#2 results: {len(reloaded)} fields, matches in-memory.")

## 9) What's next

You're done with Week 2 of R2-Q1.

You have three artifacts saved to your `OUTPUT_DIR`:

- `baseline_resnet18.pt` — your trained model.
- `eq2_results.json` — your EQ#2 deliverable.
- `training_curves.png` — loss and accuracy curves for your writeup.

**Submit EQ#2 by the end of Week 2.** Your EQ#2 writeup draws on the
test accuracy, the go/no-go verdict, and whatever else the question
page asks for (your read on whether the result matches the
Prediction's expectation of "likely exceeds 95%", any patterns you
noticed in the confusion matrix, what your training curves look like).

Week 3 picks up in Notebook 04 (`04_pv_to_pd_transfer.ipynb`). That
notebook reloads `baseline_resnet18.pt`, applies it to PlantDoc using
the class-space remapping you committed to in Notebook 02, and
measures PV → PD accuracy at your pre-committed aggregation level.
The PV-internal accuracy you just computed is the baseline for that
comparison — the gap between the two numbers is what R2-Q1 is
actually asking about.